In [1]:
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

import os
from dotenv import load_dotenv

load_dotenv()

# 1. Load pdf
documents = PyPDFLoader("./files/employee_handbook_v2.pdf").load()
print(f"Loaded documents: {len(documents)}")

# 2. Chunkerization

# 2.1 create splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)

# 2.2 create chunks
docs = text_splitter.split_documents(documents)
print (f"Created chunks: {len(docs)}")

# 3. Create DB

persist_directory = "./chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

if os.path.exists(persist_directory):
    print("Chroma directory already created.")
    # loads previously saved database
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
else:
    print("Chroma directory not yet created. Generating Vector DB.")
    # reads raw text documents, converts them to embeddings and saves to a new db
    vector_store = Chroma.from_documents(docs, embeddings, persist_directory=persist_directory)
    
print("Vector DB created/loaded.")

# 4. Create Retriever to search results, gathers 2 most relevants results for that query

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 5. Creates agent (chat model)

# temperature dictates how "creative" the llm can get
llm = init_chat_model("openai:gpt-5.4-mini", temperature=0)


/var/folders/_0/0ttx828x7156w_lylnxzl1280000gn/T/ipykernel_36592/2626699839.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded documents: 10
Created chunks: 15
Chroma directory already created.
Vector DB created/loaded.


In [5]:
# 6. Query

user_query = input("Ask away: ")
# How does the company's founding mission relate to the specific technology powering the ships, and what time must the weekly maintenance logs for that technology be submitted?

similar_docs = retriever.invoke(user_query)

llm_context = "\n\n".join(doc.page_content for doc in similar_docs) #creates a page content for each result

response = llm.invoke(f"based on this conext\n\n {llm_context}\n\n answer the following question:\n\n {user_query}")

print(response.content)


The company’s founding mission is to develop and support ships powered by the **QFD (Quantum Flux Drive)** technology, which is the core system described in the maintenance protocol.

The weekly maintenance logs for that technology must be submitted by **17:00 standard lunar time every Friday**.
